# Stage 5.05 — validate, analyze, and export

Run validation and analysis for both 5A and 5B. When the 5A0 gate is closed, 5A analysis still produces the selected operating point and the final observations without any episode results.

In [ ]:
# 5A analysis: produces the selected operating point even with an empty results file.
import os, subprocess, sys
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
ANALYSIS = OUT / "analysis"
ANALYSIS.mkdir(exist_ok=True)
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.analyze_stage5",
    "--phase", "a",
    "--results", str(OUT / "stage5a_episode_results.csv"),
    "--output-dir", str(ANALYSIS),
], cwd=R, check=True)


In [ ]:
# Build the conditional Stage 5B manifest from the selected operating point.
import json, subprocess
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
prov = json.loads((OUT / "stage5_provenance.json").read_text())
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.make_stage5b_manifest",
    "--output", str(OUT / "stage5b_manifest.csv"),
    "--selected", str(ANALYSIS / "stage5a_selected_operating_point.json"),
    "--git-sha", prov["git_sha"],
    "--libero-plus-git-sha", prov["libero_plus_git_sha"],
], cwd=R, check=True)


In [ ]:
# 5A validation (now the selected operating point exists).
import os, subprocess, sys
from pathlib import Path
R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.validate_stage5",
    "--phase", "a",
    "--manifest", str(OUT / "stage5a_manifest.csv"),
    "--output-dir", str(OUT),
    "--audit", str(OUT / "stage5_openvla_coverage_capability_audit.json"),
    "--selected", str(ANALYSIS / "stage5a_selected_operating_point.json"),
], cwd=R, check=True)


In [ ]:
# 5B validation and analysis (conditional; no-ops when the gate is closed)
import os, subprocess, sys
from pathlib import Path
R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.validate_stage5",
    "--phase", "b",
    "--manifest", str(OUT / "stage5b_manifest.csv"),
    "--output-dir", str(OUT),
    "--selected", str(ANALYSIS / "stage5a_selected_operating_point.json"),
], cwd=R, check=True)

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.analyze_stage5",
    "--phase", "b",
    "--results", str(OUT / "stage5b_episode_results.csv"),
    "--output-dir", str(ANALYSIS),
], cwd=R, check=True)


In [ ]:
import hashlib
from pathlib import Path
OUT = Path.home() / "stage5"
ANALYSIS = OUT / "analysis"

expected = ["stage5a_selected_operating_point.json", "STAGE_5A_OBSERVATIONS.md"]
missing = [x for x in expected if not (ANALYSIS / x).exists()]
if missing:
    raise SystemExit(f"missing analysis artifacts: {missing}")

print("Stage 5 analysis artifacts:")
for p in sorted(ANALYSIS.iterdir()):
    print(" ", p.name)
